In [118]:
import urllib.request
import os
import re
import glob
from bs4 import BeautifulSoup
import hashlib

Deleting empty notes

In [223]:
shelves = {
    "Biographies": "/Users/SFL/notes/content/Other Genres/Biographies",
    "Business Thinking": "/Users/SFL/notes/content/Self-help/Business Thinking",
    "Fiction": "/Users/SFL/notes/content/Other Genres/Fiction",
    "Humanities": "/Users/SFL/notes/content/Other Genres/Humanities",
    "Personal Development": "/Users/SFL/notes/content/Self-help/Personal Development",
    "Natural & Social Sciences": "/Users/SFL/notes/content/Natural & Social Sciences"
}

exclude_files = {
    "Natural & Social Sciences.md",
    "Personal Development.md",
    "Business Thinking.md",
    "Biographies.md",
    "Fiction.md",
    "Humanities.md",
    "index.md"
}

def is_empty_or_generated_only(file_path):
    # return True
    """Checks if a file contains only auto-generated content (subtitles) and no actual written content."""
    try:
        with open(file_path, "r", encoding="utf-8") as file:
            content = file.read().strip()
            content = re.sub(r"^---.*?---", "", content, flags=re.DOTALL).strip()
            subtitle_pattern = re.compile(r"^## — .+", re.MULTILINE)
            content_without_subtitles = subtitle_pattern.sub("", content).strip()

            return len(content_without_subtitles) == 0

    except Exception as e:
        print(f"Error reading {file_path}: {e}")
        return False  # Default to not deleting if there's an error

def clear_folder(folder_path):
    """Deletes only notes that are empty or contain only generated subtitles."""
    if os.path.exists(folder_path):
        for filename in os.listdir(folder_path):
            if filename in exclude_files:
                continue
            
            file_path = os.path.join(folder_path, filename)

            try:
                if os.path.isfile(file_path) and filename.endswith(".md"):
                    if is_empty_or_generated_only(file_path):
                        os.remove(file_path)
                    else:
                        print(f"Kept: {os.path.splitext(filename)[0]}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")
    else:
        print(f"Folder does not exist: {folder_path}")

for category, folder_path in shelves.items():
    clear_folder(folder_path)

print("Cleanup completed!")

Cleanup completed!


Creating book links

In [231]:
def update_shelf_notes(shelves):
    for shelf_name, folder_path in shelves.items():
        main_note_path = os.path.join(folder_path, "index.md")  # Ensure matching case

        if not os.path.isfile(main_note_path):
            
            # Create a new file named 'index.md' at the same directory
            index_note_path = os.path.join(os.path.dirname(main_note_path), "index.md")
            
            with open(index_note_path, "w", encoding="utf-8") as f:
                f.write(f"""---
title: {shelf_name.capitalize()}
---

""")  # Initial content
            print(f"Created new file: {index_note_path}")


        # Collect titles of all other .md files in the folder
        book_titles = [
            os.path.splitext(filename)[0]  # Remove .md extension
            for filename in os.listdir(folder_path)
            if filename.endswith(".md") and filename != "index.md"  # Exclude the main note
        ]

        # Format book titles as Obsidian links (sorted A-Z)
        book_links = "\n".join([f"[[{title}]]" for title in sorted(book_titles)])

        # Update the main note
        try:
            with open(main_note_path, "r", encoding="utf-8") as file:
                content = file.read()

            # Find the marker string and truncate everything after it
            marker = "**Full List of Books in this Category A-Z**"
            if marker in content:
                content = content.split(marker)[0] + marker + "\n" + book_links  # No extra newline
            else:
                content += f"\n\n{marker}\n" + book_links  # Ensure marker exists, no extra line

            # Write the updated content back to the main note
            with open(main_note_path, "w", encoding="utf-8") as file:
                file.write(content)

            print(f"Updated note: {main_note_path}")
        except Exception as e:
            print(f"Error updating {main_note_path}: {e}")

# Define the corrected shelf directories
shelves = {
    "Biographies": "/Users/SFL/notes/content/Other Genres/Biographies",
    "Business Thinking": "/Users/SFL/notes/content/Self-help/Business Thinking",
    "Fiction": "/Users/SFL/notes/content/Other Genres/Fiction",
    "Humanities": "/Users/SFL/notes/content/Other Genres/Humanities",
    "Personal Development": "/Users/SFL/notes/content/Self-help/Personal Development",
    "Natural & Social Sciences": "/Users/SFL/notes/content/Natural & Social Sciences"
}

# Run the function
update_shelf_notes(shelves)

Updated note: /Users/SFL/notes/content/Other Genres/Biographies/index.md
Updated note: /Users/SFL/notes/content/Self-help/Business Thinking/index.md
Created new file: /Users/SFL/notes/content/Other Genres/Fiction/index.md
Updated note: /Users/SFL/notes/content/Other Genres/Fiction/index.md
Updated note: /Users/SFL/notes/content/Other Genres/Humanities/index.md
Updated note: /Users/SFL/notes/content/Self-help/Personal Development/index.md
Updated note: /Users/SFL/notes/content/Natural & Social Sciences/index.md


Importing highlights

In [229]:
OBSIDIAN_DIR = "/Users/SFL/notes/content"

def generate_hash(text):
    """Generates a short hash for a given text."""
    return hashlib.md5(text.encode()).hexdigest()[:6]

def extract_book_title_and_quotes(html_path):
    """
    Extracts the book title, section headings, chapter headings, and formatted highlights
    from a Kindle HTML file.

    Section headings (<div class="sectionHeading">) are added as new lines starting with "### ",
    with any square brackets [] or double quotes removed and the text converted to title case.

    For note headings (<div class="noteHeading">) that start with "Highlight", the chapter
    heading is extracted (from the text following " - " and before " > ") and, if it is different
    from the previously seen chapter, added as a new line starting with "#### " (again cleaned and capitalized).

    The actual highlight text comes from the following <div class="noteText"> element.
    Nested notes (when the note heading starts with "Note") are appended to the last highlight.
    """
    with open(html_path, "r", encoding="utf-8") as file:
        soup = BeautifulSoup(file, "html.parser")

    # Get the book title from <div class="bookTitle">
    book_title_elem = soup.find("div", class_="bookTitle")
    book_title = book_title_elem.get_text(strip=True) if book_title_elem else None

    # Process all elements in document order that could be a section heading, note heading, or note text.
    elements = soup.find_all("div", class_=["sectionHeading", "noteHeading", "noteText"])
    
    content_lines = []
    pending_meta = None  # Holds the meta info (from noteHeading) until we see a noteText
    current_chapter = None  # Stores the current chapter heading so that repeated ones are not re-added

    for elem in elements:
        classes = elem.get("class", [])
        if "sectionHeading" in classes:
            # Process a section heading: clean and prepend with "### "
            section_text = elem.get_text(strip=True)
            section_text = re.sub(r'[\[\]"]', '', section_text)  # Remove [ ] and " characters
            section_text = section_text.title()  # Convert to title case
            content_lines.append("### " + section_text)
        elif "noteHeading" in classes:
            meta_info = elem.get_text(strip=True)
            if meta_info.startswith("Highlight"):
                # Attempt to extract a chapter heading from the meta text.
                # For example, from:
                #   "Highlight(<span class="highlight_yellow">yellow</span>) - 1. The Baboons: The Generations of Israel > Page 18 · Location 146"
                # we split on " - " then on " > " to extract the chapter part.
                parts = meta_info.split(" - ", 1)
                if len(parts) > 1:
                    remainder = parts[1]
                    chapter = remainder.split(" > ")[0].strip()
                    chapter_clean = re.sub(r'[\[\]"]', '', chapter)
                    chapter_clean = chapter_clean.title()
                    # Only add the chapter heading if it has changed
                    if current_chapter != chapter_clean:
                        content_lines.append("#### " + chapter_clean)
                        current_chapter = chapter_clean
                # Set pending meta to later pair with the following noteText
                pending_meta = {"type": "highlight", "meta": meta_info}
            elif meta_info.startswith("Note"):
                # This is a nested note for the last highlight.
                pending_meta = {"type": "note", "meta": meta_info}
        elif "noteText" in classes:
            text = elem.get_text(strip=True)
            if pending_meta:
                if pending_meta["type"] == "highlight":
                    # Look for page information in the meta info
                    page_match = re.search(r"Page (\d+)", pending_meta["meta"])
                    page_info = f" – Page {page_match.group(1)}" if page_match else ""
                    formatted_quote = f"> {text}{page_info}"
                    content_lines.append(formatted_quote)
                elif pending_meta["type"] == "note":
                    # Append nested note text to the last highlight.
                    formatted_note = f"> > =={text}=="
                    # Search backwards for the last highlight (line starting with "> ")
                    for idx in range(len(content_lines) - 1, -1, -1):
                        if content_lines[idx].startswith("> "):
                            content_lines[idx] += "\n" + formatted_note
                            break
                pending_meta = None  # Clear pending meta after processing noteText

    return book_title, content_lines

def find_matching_note(book_title):
    """Finds the best-matching note based on the beginning of the HTML title."""
    b = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", book_title)
    note_files = glob.glob(f"{OBSIDIAN_DIR}/**/*.md", recursive=True)
    note_titles = {os.path.splitext(os.path.basename(f))[0]: f for f in note_files}

    for note_title in note_titles:
        n = re.sub(r"[^\u4e00-\u9fffA-Za-z0-9]", "", note_title)
        if b.lower().startswith(n.lower()):
            return note_titles[note_title]

    return None

def parse_existing_highlights(note_path):
    """Parses existing highlights and creates hash mappings for notes & identifiers."""
    with open(note_path, "r", encoding="utf-8") as file:
        content = file.read()

    in_select_quotes = False
    preserved_hashes = set()
    highlight_extra_map = {}

    lines = content.splitlines()
    lines = [s for s in lines if s.strip()]
    
    for i, line in enumerate(lines):
        if "## Select Quotes" in line:
            in_select_quotes = True
            continue

        if in_select_quotes:
            if line.startswith("> ") and not line.startswith("> >"):
                hash_val = generate_hash(line)
                has_note = bool(i + 1 < len(lines) and lines[i + 1].strip().startswith("> >"))
                has_id = bool(i + 1 < len(lines) and re.match(r"^\^[a-zA-Z0-9]{6}$", lines[i + 1].strip()))
                has_both = bool(i + 2 < len(lines) and lines[i + 1].strip().startswith("> >") and re.match(r"^\^[a-zA-Z0-9]{6}$", lines[i + 2].strip()))

                if has_both:
                    preserved_hashes.add(hash_val)
                    highlight_extra_map[hash_val] = lines[i + 1] + '\n' + lines[i + 2]
                elif has_note or has_id:
                    preserved_hashes.add(hash_val)
                    highlight_extra_map[hash_val] = lines[i + 1]
                
    return preserved_hashes, highlight_extra_map

def update_obsidian_note(note_path, highlights):
    """
    Updates the given Obsidian note with extracted highlights while preserving
    manually added notes and identifiers.
    """
    preserved_hashes, highlight_extra_map = parse_existing_highlights(note_path)

    with open(note_path, "r", encoding="utf-8") as file:
        content = file.read()

    if "## Select Quotes" in content:
        content = re.sub(r"## Select Quotes\n.*", "## Select Quotes", content, flags=re.DOTALL).strip()
    else:
        content += "\n\n## Select Quotes"

    new_content = []

    for i, line in enumerate(highlights):
        if line.strip().startswith("###"):
            new_content.append('\n' + line)
        elif line.strip().startswith("> ") and not line.startswith("> >"):
            new_content.append('\n' + line)
            highlight_hash = generate_hash(line)
            if highlight_hash in preserved_hashes:
                new_content.append(highlight_extra_map[highlight_hash]) 
        elif line.startswith("> >"):
            previous_hash = generate_hash(highlights[i - 1])
            if previous_hash in preserved_hashes and '> >' in highlight_extra_map[previous_hash]:
                continue
            else:
                new_content.append(line)

    new_content = "\n".join(new_content) + "\n"

    with open(note_path, "w", encoding="utf-8") as file:
        file.write(content + '\n' + new_content)

def process_kindle_highlights(html_path):
    """
    Extracts Kindle highlights from the provided HTML file and updates the
    corresponding Obsidian note.
    """
    book_title, highlights = extract_book_title_and_quotes(html_path)

    if not book_title:
        print(f"Could not extract book title from {html_path}")
        return

    note_path = find_matching_note(book_title)
    
    if not note_path:
        print(f"No matching Obsidian note found for: {book_title}")
        return

    update_obsidian_note(note_path, highlights)

In [230]:
HIGHLIGHTS_DIR = "/Users/SFL/notes/private/highlights"

def process_all_highlights():
    # Iterate over all files in the directory
    for file_name in os.listdir(HIGHLIGHTS_DIR):
        file_path = os.path.join(HIGHLIGHTS_DIR, file_name)
        
        # Process only files (skip directories or invalid files)
        if os.path.isfile(file_path) and file_name.endswith(".html"):
            process_kindle_highlights(file_path)

# Run the function
process_all_highlights()

No matching Obsidian note found for: Gu Du Liu Jiang
No matching Obsidian note found for: Sheng Huo Shi Jiang


In [233]:
process_kindle_highlights('/Users/SFL/Downloads/The Gifts of Imperfection_ Let Go of Who Y - Brene Brown - Notebook.html')

In [14]:
def process_eml_files(folder_path, output_folder):
    """
    Process all .eml files in the folder, combining highlights from the same book,
    removing duplicates, and sorting by timestamp.
    """
    # Dictionary to store highlights by book name
    book_highlights = defaultdict(list)
    
    # Process all .eml files
    for filename in os.listdir(folder_path):
        if not filename.endswith('.eml'):
            continue
            
        # Extract book name (everything before _)
        book_name = re.split(r'[_.]', filename)[0].strip()
        
        try:
            # Read and parse .eml file
            with open(os.path.join(folder_path, filename), 'rb') as f:
                msg = email.message_from_binary_file(f, policy=policy.default)
            
            # Get email body
            body = ""
            if msg.is_multipart():
                for part in msg.walk():
                    if part.get_content_type() == "text/plain":
                        body = part.get_payload(decode=True).decode()
                        break
            else:
                body = msg.get_payload(decode=True).decode()
            
            # Parse highlights
            lines = body.split('\n')
            i = 0
            while i < len(lines):
                line = lines[i].strip()
                
                # Print lines starting with '注'
                if line.startswith('注'):
                    print(f"Found note in {filename}: {line}")
                
                # Look for timestamp pattern
                timestamp_match = re.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$', line)
                if timestamp_match and i + 1 < len(lines):
                    timestamp = datetime.strptime(line, '%Y-%m-%d %H:%M:%S')
                    highlight_text = lines[i + 1].strip()
                    
                    if highlight_text:  # Skip empty highlights
                        # Also check if highlight text starts with '注'
                        if highlight_text.startswith('注'):
                            print(f"Found note in {filename}: {highlight_text}")
                            
                        highlight_hash = hashlib.md5(highlight_text.encode()).hexdigest()
                        book_highlights[book_name].append({
                            'timestamp': timestamp,
                            'text': highlight_text,
                            'hash': highlight_hash
                        })
                    i += 2
                else:
                    i += 1
                    
        except Exception as e:
            print(f"Error processing {filename}: {e}")
    
    # Rest of the function remains the same...
    for book_name, highlights in book_highlights.items():
        highlights.sort(key=lambda x: x['timestamp'])
        
        seen_hashes = set()
        unique_highlights = []
        for h in highlights:
            if h['hash'] not in seen_hashes:
                seen_hashes.add(h['hash'])
                unique_highlights.append(h)
        
        output_filename = f"{book_name.title()}.md"
        output_path = os.path.join(output_folder, output_filename)
        
        try:
            with open(output_path, 'w', encoding='utf-8') as f:
                f.write(f"# {book_name.title()}\n\n")
                f.write("## Select Quotes\n\n")
                
                for highlight in unique_highlights:
                    timestamp_str = highlight['timestamp'].strftime('%Y-%m-%d %H:%M:%S')
                    f.write(f"> {highlight['text']}\n")
                    f.write(f"^{highlight['hash'][:6]}\n\n")
            
            print(f"Created: {output_filename}")
            
        except Exception as e:
            print(f"Error writing {output_filename}: {e}")

# Example usage
eml_folder = "/Users/SFL/Desktop"
output_folder = "/Users/SFL/Desktop/outputs"
process_eml_files(eml_folder, output_folder)

Found note in  Minor Feelings_1.eml: 注: 有百年孤独那味儿了
Created: 政治学通识.md
Created: The Art Of Loving.md
Created: Why We Sleep.md
Created: Steve Jobs.md
Created: The Miracle Morning.md
Created: The Real Happy Pill.md
Created: Minor Feelings.md
Created: Atomic Habits.md
